In [ ]:
import sys
import re
import os
from datetime import datetime
from PyQt6.QtWidgets import (QMainWindow, QApplication, QListWidget,
                             QFileDialog, QMessageBox, QLabel)
from PyQt6.QtGui import QAction


class StringFinderApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.log_file = "script18.log"
        self.initUI()
        self.check_log_exists()

    def initUI(self):
        self.setWindowTitle('Скрипт 18. Искатель строк')
        self.setGeometry(100, 100, 800, 500)

        self.result_list = QListWidget()
        self.setCentralWidget(self.result_list)

        menubar = self.menuBar()

        file_menu = menubar.addMenu('Файл')
        open_action = QAction('Открыть...', self)
        open_action.triggered.connect(self.open_file)
        file_menu.addAction(open_action)

        log_menu = menubar.addMenu('Лог')
        export_action = QAction('Экспорт...', self)
        export_action.triggered.connect(self.export_log)
        add_log_action = QAction('Добавить в лог', self)
        add_log_action.triggered.connect(self.add_to_log)
        view_log_action = QAction('Просмотр', self)
        view_log_action.triggered.connect(self.view_log)

        log_menu.addActions([export_action, add_log_action, view_log_action])

        self.statusBar = self.statusBar()
        self.status_msg = QLabel("Готов к работе")
        self.status_size = QLabel("")

        self.statusBar.addPermanentWidget(self.status_msg, 6)
        self.statusBar.addPermanentWidget(self.status_size, 4)

    def check_log_exists(self):
        if not os.path.exists(self.log_file):
            QMessageBox.information(self, "Инфо", "Файл лога не найден. Файл будет создан автоматически.")
            open(self.log_file, 'a').close()

    def open_file(self):
        fname, _ = QFileDialog.getOpenFileName(self, 'Открыть файл')
        if fname:
            pattern = r'\d{2}-\d{2}-\d{4}'
            search_time = datetime.now().strftime("%d.%m.%Y %H:%M:%S")

            self.result_list.addItem(f"Файл {fname} был обработан {search_time}:")

            try:
                with open(fname, 'r', encoding='utf-8') as f:
                    for line_num, line in enumerate(f, 1):
                        for match in re.finditer(pattern, line):
                            res = f"Строка {line_num}, позиция {match.start() + 1} : найдено «{match.group()}»"
                            self.result_list.addItem(res)

                file_size = os.path.getsize(fname)
                formatted_size = "{:,} байт".format(file_size).replace(",", " ")
                self.status_msg.setText(f"Обработан файл {os.path.basename(fname)}")
                self.status_size.setText(formatted_size)

            except Exception as e:
                QMessageBox.critical(self, "Ошибка", f"Не удалось прочитать файл: {e}")

    def add_to_log(self):
        with open(self.log_file, 'a', encoding='utf-8') as f:
            for i in range(self.result_list.count()):
                f.write(self.result_list.item(i).text() + "\n")
        self.status_msg.setText("Данные добавлены в лог")

    def export_log(self):
        fname, _ = QFileDialog.getSaveFileName(self, 'Экспорт лога', filter="Text files (*.txt)")
        if fname:
            with open(fname, 'w', encoding='utf-8') as f:
                for i in range(self.result_list.count()):
                    f.write(self.result_list.item(i).text() + "\n")

    def view_log(self):
        reply = QMessageBox.question(self, 'Вопрос',
                                     "Вы действительно хотите открыть лог? Данные последних поисков будут потеряны!",
                                     QMessageBox.StandardButton.Yes | QMessageBox.StandardButton.No)

        if reply == QMessageBox.StandardButton.Yes:
            self.result_list.clear()
            if os.path.exists(self.log_file):
                with open(self.log_file, 'r', encoding='utf-8') as f:
                    for line in f:
                        self.result_list.addItem(line.strip())
                self.status_msg.setText("Открыт лог")


if __name__ == '__main__':
    app = QApplication(sys.argv)
    ex = StringFinderApp()
    ex.show()
    sys.exit(app.exec())
